# Final XGBoost Design: Jeonse-Equivalent Price Prediction with SHAP

This notebook reflects the post-meeting design decision:

- Use **XGBoost only** as the final model.
- Use monthly-rent rows by converting them into Jeonse-equivalent deposits.
- Compare feature sets rather than comparing multiple model families.
- Use SHAP to check whether spatial-derived variables meaningfully contribute to the final XGBoost prediction.

Target variable: `전세환산보증금(만원)`.

## 1. Final Experimental Design

The final report should not present Ridge, RandomForest, and XGBoost as competing final models. XGBoost is selected as the final model because it produced the best predictive accuracy.

The experiment is reframed as a feature-set comparison using the same XGBoost model:

| Experiment | Purpose | Features |
|---|---|---|
| `A_baseline_no_dong` | Basic baseline | apartment structure + contract time |
| `B_spatial_no_dong` | Spatial added-value without administrative location control | baseline + sea/park/school variables |
| `A_location_baseline` | Stronger baseline with administrative location control | baseline + `읍면동` |
| `B_location_spatial` | Final performance model | baseline + `읍면동` + spatial variables |

Interpretation rule:

- Main performance model: `B_location_spatial`.
- Spatial added-value check: compare `A_baseline_no_dong` vs `B_spatial_no_dong`.
- Conservative location-control check: compare `A_location_baseline` vs `B_location_spatial`.
- SHAP interpretation: if sea/park/school groups have meaningful SHAP importance, the spatial-variable argument is supported; otherwise, the paper should emphasize prediction performance more than spatial interpretation.

## 2. Colab Setup

Run this cell first. It clones the review branch when the dataset is not already present and installs only missing packages.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import importlib.util

REPO_URL = "https://github.com/hyeon03-sketch/IML-Final-project.git"
REPO_BRANCH = "codex/ml-project-review"
REPO_DIR = Path("IML-Final-project")
DATA_FILE = Path("IML_Final_dataset.xlsx")

if not DATA_FILE.exists() and Path("../IML_Final_dataset.xlsx").exists():
    os.chdir("..")

if not DATA_FILE.exists():
    if not REPO_DIR.exists():
        subprocess.check_call(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL])
    os.chdir(REPO_DIR)

print("Working directory:", Path.cwd())
print("Dataset exists:", Path("IML_Final_dataset.xlsx").exists())

packages = {
    "pandas": "pandas",
    "numpy": "numpy",
    "openpyxl": "openpyxl",
    "sklearn": "scikit-learn",
    "xgboost": "xgboost",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "shap": "shap",
}
missing = [pip_name for import_name, pip_name in packages.items() if importlib.util.find_spec(import_name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("Installed:", missing)
else:
    print("All required packages are already installed.")

## 3. Imports and Korean Font Setup

In [ ]:
import warnings
from pathlib import Path
import subprocess

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import shap

from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, GroupShuffleSplit, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
RND = 42
DATA_PATH = Path("IML_Final_dataset.xlsx")
SHEET_NAME = "최종데이터셋_모델용"
TARGET = "전세환산보증금(만원)"
CONVERSION_RATE = 0.065
OUT_DIR = Path("outputs_final_xgboost_shap")
OUT_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)


def setup_korean_font():
    font_candidates = [
        ("NanumGothic", Path("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")),
        ("Apple SD Gothic Neo", Path("/System/Library/Fonts/AppleSDGothicNeo.ttc")),
        ("AppleGothic", Path("/Library/Fonts/AppleGothic.ttf")),
        ("Malgun Gothic", Path("C:/Windows/Fonts/malgun.ttf")),
    ]
    for font_name, font_path in font_candidates:
        if font_path.exists():
            try:
                fm.fontManager.addfont(str(font_path))
            except Exception:
                pass
            mpl.rcParams["font.family"] = font_name
            mpl.rcParams["axes.unicode_minus"] = False
            return font_name
    try:
        subprocess.check_call(["apt-get", "update", "-qq"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        subprocess.check_call(["apt-get", "install", "-y", "fonts-nanum"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        fm._load_fontmanager(try_read_cache=False)
        mpl.rcParams["font.family"] = "NanumGothic"
        mpl.rcParams["axes.unicode_minus"] = False
        return "NanumGothic"
    except Exception:
        mpl.rcParams["axes.unicode_minus"] = False
        return "default"

font_used = setup_korean_font()
print("Font:", font_used)

## 4. Load Data and Create Jeonse-Equivalent Target

Monthly-rent rows are retained by converting monthly rent into a Jeonse-equivalent deposit:

$$
\text{Jeonse-equivalent deposit} = \text{Deposit} + \frac{\text{Monthly rent} \times 12}{\text{Conversion rate}}
$$

The default conversion rate is 6.5% (`0.065`). Deposit and monthly rent are both measured in 10,000 KRW units, so the converted target is also in 10,000 KRW units.

In [ ]:
def make_jeonse_equivalent_target(data, conversion_rate=CONVERSION_RATE):
    out = data.copy()
    deposit = pd.to_numeric(out["보증금(만원)"], errors="coerce")
    monthly_rent = pd.to_numeric(out["월세금(만원)"], errors="coerce").fillna(0)
    lease_type = out["전월세구분"].astype(str).str.strip()

    target = deposit.astype(float).copy()
    monthly_mask = lease_type.eq("월세")
    target.loc[monthly_mask] = deposit.loc[monthly_mask] + monthly_rent.loc[monthly_mask] * 12.0 / conversion_rate
    out[TARGET] = target
    return out

raw = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)
df = make_jeonse_equivalent_target(raw, CONVERSION_RATE)
df = df[df[TARGET].notna() & df[TARGET].gt(0)].copy()

print("Original rows:", len(raw))
print("Modeling rows:", len(df))
print("Jeonse rows:", int(df["전월세구분"].eq("전세").sum()))
print("Monthly-rent rows converted:", int(df["전월세구분"].eq("월세").sum()))
print("Dong count:", df["읍면동"].nunique())
print("Contract years:", sorted(df["계약연도"].dropna().unique()))
print("Missing cells in selected raw data:", int(df.isna().sum().sum()))

display(df[["전월세구분", "보증금(만원)", "월세금(만원)", TARGET]].head(10))
display(df[[TARGET, "전용면적(㎡)", "층", "건물연령(계약기준)", "계약연도", "계약월"]].describe())

## 5. Target Distribution

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df[TARGET], bins=40, kde=True, color="#2563eb")
plt.title("Target Distribution: Jeonse-Equivalent Deposit")
plt.xlabel("Jeonse-equivalent deposit (만원)")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(OUT_DIR / "target_distribution.png", dpi=150)
plt.show()

print("Target mean:", round(df[TARGET].mean(), 2))
print("Target median:", round(df[TARGET].median(), 2))
print("Target std:", round(df[TARGET].std(), 2))

## 6. Feature Sets and Dong-Level Collinearity Check

Spatial variables are administrative-dong-level variables. If they are perfectly determined by `읍면동`, then including both `읍면동` one-hot variables and spatial variables can split attribution between them. This is not direct target leakage, but it matters for interpretation.

In [ ]:
BASE_FEATURES = ["전용면적(㎡)", "층", "건물연령(계약기준)", "계약연도", "계약월"]
DONG_FEATURE = ["읍면동"]
SPATIAL_FEATURES = [
    "바다여부",
    "공원여부",
    "공원수",
    "최대공원면적(㎡)",
    "동_초등학교수",
    "동_중학교수",
    "동_고등학교수",
    "동_초중고모두있음여부",
]

# Removed from the default design because of clear redundancy or high overlap:
# - 동_총학교수 = 동_초등학교수 + 동_중학교수 + 동_고등학교수
# - 총공원면적(㎡) overlaps strongly with other park area/count variables in previous VIF checks.
EXCLUDED_SPATIAL_FEATURES = ["동_총학교수", "총공원면적(㎡)"]

FEATURE_GROUPS = {
    "A_baseline_no_dong": BASE_FEATURES,
    "B_spatial_no_dong": BASE_FEATURES + SPATIAL_FEATURES,
    "A_location_baseline": BASE_FEATURES + DONG_FEATURE,
    "B_location_spatial": BASE_FEATURES + DONG_FEATURE + SPATIAL_FEATURES,
}

BINARY_COLS = {"바다여부", "공원여부", "동_초중고모두있음여부"}
CATEGORICAL_COLS = {"읍면동"}

print("Feature groups")
for name, cols in FEATURE_GROUPS.items():
    print(name, len(cols), cols)

# Check whether spatial variables are constant within each dong.
dong_level_rows = []
for col in SPATIAL_FEATURES + EXCLUDED_SPATIAL_FEATURES:
    max_unique_by_dong = int(df.groupby("읍면동")[col].nunique(dropna=False).max())
    dong_level_rows.append({
        "feature": col,
        "max_unique_values_within_dong": max_unique_by_dong,
        "fully_determined_by_dong": max_unique_by_dong == 1,
    })
dong_level_check = pd.DataFrame(dong_level_rows)
display(dong_level_check)
dong_level_check.to_csv(OUT_DIR / "dong_level_spatial_check.csv", index=False, encoding="utf-8-sig")

## 7. XGBoost Pipeline and Metrics

No scaler is used because XGBoost is tree-based and does not require feature scaling. Categorical `읍면동` is one-hot encoded only in the location-control settings.

In [ ]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def split_feature_types(cols):
    categorical = [c for c in cols if c in CATEGORICAL_COLS]
    binary = [c for c in cols if c in BINARY_COLS]
    numeric = [c for c in cols if c not in set(categorical + binary)]
    return numeric, categorical, binary


def make_preprocessor(cols):
    numeric, categorical, binary = split_feature_types(cols)
    transformers = []
    pass_cols = numeric + binary
    if pass_cols:
        transformers.append(("pass", "passthrough", pass_cols))
    if categorical:
        transformers.append(("cat", make_one_hot_encoder(), categorical))
    return ColumnTransformer(transformers=transformers, remainder="drop")


def make_xgboost():
    return XGBRegressor(
        objective="reg:squarederror",
        tree_method="hist",
        random_state=RND,
        n_jobs=-1,
        importance_type="gain",
    )

XGB_PARAM_GRID = {
    "model__n_estimators": [300, 600],
    "model__max_depth": [4, 6],
    "model__learning_rate": [0.05, 0.1],
    "model__subsample": [0.9],
    "model__colsample_bytree": [0.9],
}


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def mape_pct(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = y_true > 0
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)


def evaluate_predictions(y_true, pred):
    return {
        "test_rmse": rmse(y_true, pred),
        "test_mae": float(mean_absolute_error(y_true, pred)),
        "test_mape_pct": mape_pct(y_true, pred),
        "test_r2": float(r2_score(y_true, pred)),
        "mae_pct_of_mean_target": float(mean_absolute_error(y_true, pred) / y_true.mean() * 100),
    }


def fit_xgb_grid(data, group_name, cols, train_idx, test_idx):
    X = data[cols].copy()
    y = data[TARGET].copy()
    X_train, X_test = X.loc[train_idx], X.loc[test_idx]
    y_train, y_test = y.loc[train_idx], y.loc[test_idx]

    pipe = Pipeline([
        ("pre", make_preprocessor(cols)),
        ("model", make_xgboost()),
    ])
    search = GridSearchCV(
        pipe,
        param_grid=XGB_PARAM_GRID,
        scoring="neg_root_mean_squared_error",
        cv=KFold(n_splits=5, shuffle=True, random_state=RND),
        n_jobs=-1,
        verbose=0,
    )
    search.fit(X_train, y_train)
    pred = search.predict(X_test)
    metrics = evaluate_predictions(y_test, pred)
    metrics.update({
        "experiment": group_name,
        "model": "XGBoost",
        "n_train": len(train_idx),
        "n_test": len(test_idx),
        "n_features_raw": len(cols),
        "cv_rmse": float(-search.best_score_),
        "best_params": search.best_params_,
    })
    fitted = {
        "estimator": search.best_estimator_,
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "pred": pred,
        "best_params": search.best_params_,
    }
    return metrics, fitted

## 8. Main XGBoost Feature-Set Experiments

In [ ]:
train_idx, test_idx = train_test_split(df.index, test_size=0.2, random_state=RND)

rows = []
fitted = {}
for group_name, cols in FEATURE_GROUPS.items():
    print("Running", group_name)
    metrics, bundle = fit_xgb_grid(df, group_name, cols, train_idx, test_idx)
    rows.append(metrics)
    fitted[group_name] = bundle
    print(
        f"  RMSE={metrics['test_rmse']:.2f}, MAE={metrics['test_mae']:.2f}, "
        f"MAPE={metrics['test_mape_pct']:.2f}%, R2={metrics['test_r2']:.4f}"
    )

results = pd.DataFrame(rows).sort_values("test_rmse")
display(results[[
    "experiment", "model", "n_train", "n_test", "n_features_raw", "cv_rmse",
    "test_rmse", "test_mae", "test_mape_pct", "test_r2", "mae_pct_of_mean_target", "best_params"
]])
results.to_csv(OUT_DIR / "xgboost_feature_set_results.csv", index=False, encoding="utf-8-sig")

## 9. A/B Comparison Tables

These comparisons answer two different questions.

1. No-dong comparison: Do spatial variables improve the baseline model?
2. Location-control comparison: Do spatial variables add value after administrative dong is already included?

In [ ]:
def compare_delta(left, right):
    left_row = results[results["experiment"].eq(left)].iloc[0]
    right_row = results[results["experiment"].eq(right)].iloc[0]
    out = pd.DataFrame([{
        "comparison": f"{right} - {left}",
        "delta_rmse": right_row["test_rmse"] - left_row["test_rmse"],
        "delta_mae": right_row["test_mae"] - left_row["test_mae"],
        "delta_mape_pct": right_row["test_mape_pct"] - left_row["test_mape_pct"],
        "delta_r2": right_row["test_r2"] - left_row["test_r2"],
    }])
    return out

comparison = pd.concat([
    compare_delta("A_baseline_no_dong", "B_spatial_no_dong"),
    compare_delta("A_location_baseline", "B_location_spatial"),
], ignore_index=True)

display(comparison)
comparison.to_csv(OUT_DIR / "xgboost_ab_comparison.csv", index=False, encoding="utf-8-sig")

best_row = results.sort_values("test_rmse").iloc[0]
print("Best XGBoost setting:", best_row["experiment"])
print(f"RMSE={best_row['test_rmse']:.2f}, MAE={best_row['test_mae']:.2f}, MAPE={best_row['test_mape_pct']:.2f}%, R2={best_row['test_r2']:.4f}")

## 10. Prediction vs Actual for Final Model

The final performance model is set to `B_location_spatial`, because it includes apartment characteristics, contract time, administrative location, and spatial-derived variables.

In [ ]:
FINAL_GROUP = "B_location_spatial"
final_bundle = fitted[FINAL_GROUP]

plt.figure(figsize=(6, 6))
plt.scatter(final_bundle["y_test"], final_bundle["pred"], s=10, alpha=0.45, color="#2563eb")
lim = max(final_bundle["y_test"].max(), np.max(final_bundle["pred"]))
plt.plot([0, lim], [0, lim], "r--", lw=1)
plt.xlabel("Actual Jeonse-equivalent deposit (만원)")
plt.ylabel("Predicted Jeonse-equivalent deposit (만원)")
plt.title(f"Prediction vs Actual: {FINAL_GROUP}")
plt.tight_layout()
plt.savefig(OUT_DIR / "prediction_vs_actual_final.png", dpi=150)
plt.show()

## 11. SHAP Interpretation

SHAP is used to determine whether spatial-derived variables are important in the final XGBoost model. Because `읍면동` and spatial variables may share location information, both feature-level and grouped SHAP importance are reported.

In [ ]:
SHAP_SAMPLE_SIZE = 800

pipe = final_bundle["estimator"]
pre = pipe.named_steps["pre"]
model = pipe.named_steps["model"]

X_test = final_bundle["X_test"].copy()
if len(X_test) > SHAP_SAMPLE_SIZE:
    X_shap = X_test.sample(SHAP_SAMPLE_SIZE, random_state=RND)
else:
    X_shap = X_test

X_shap_transformed = pre.transform(X_shap)
feature_names = pre.get_feature_names_out()

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_shap_transformed)

shap_importance = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0),
}).sort_values("mean_abs_shap", ascending=False)


def feature_group(feature_name):
    name = str(feature_name)
    if "읍면동" in name:
        return "dong_location"
    if "바다" in name:
        return "sea"
    if "공원" in name:
        return "park"
    if "학교" in name or "초중고" in name:
        return "school"
    if "계약" in name:
        return "contract_time"
    return "housing_structure"

shap_importance["feature_group"] = shap_importance["feature"].map(feature_group)
group_shap = (
    shap_importance.groupby("feature_group", as_index=False)["mean_abs_shap"]
    .sum()
    .sort_values("mean_abs_shap", ascending=False)
)
group_shap["share_pct"] = group_shap["mean_abs_shap"] / group_shap["mean_abs_shap"].sum() * 100

spatial_groups = {"sea", "park", "school"}
spatial_share = group_shap[group_shap["feature_group"].isin(spatial_groups)]["share_pct"].sum()

print(f"Spatial SHAP share (sea + park + school): {spatial_share:.2f}%")
display(shap_importance.head(30))
display(group_shap)

shap_importance.to_csv(OUT_DIR / "shap_feature_importance.csv", index=False, encoding="utf-8-sig")
group_shap.to_csv(OUT_DIR / "shap_group_importance.csv", index=False, encoding="utf-8-sig")

plt.figure(figsize=(9, 7))
top = shap_importance.head(20).iloc[::-1]
plt.barh(top["feature"], top["mean_abs_shap"], color="#059669")
plt.title("SHAP Feature Importance: Final XGBoost")
plt.xlabel("Mean absolute SHAP value")
plt.tight_layout()
plt.savefig(OUT_DIR / "shap_top20_features.png", dpi=150)
plt.show()

plt.figure(figsize=(7, 4))
sns.barplot(data=group_shap, y="feature_group", x="share_pct", color="#2563eb")
plt.title("Grouped SHAP Importance Share")
plt.xlabel("Share of total mean absolute SHAP (%)")
plt.ylabel("")
plt.tight_layout()
plt.savefig(OUT_DIR / "shap_group_importance.png", dpi=150)
plt.show()

## 12. SHAP Summary Plot

In [ ]:
# SHAP summary plots can be visually dense, but they are useful for checking direction and magnitude.
shap.summary_plot(shap_values, X_shap_transformed, feature_names=feature_names, show=False, max_display=20)
plt.tight_layout()
plt.savefig(OUT_DIR / "shap_summary_top20.png", dpi=150, bbox_inches="tight")
plt.show()

## 13. Robustness Validation

This section checks whether the final XGBoost feature-set comparison is maintained under stricter splits.

Default strict comparison: `A_location_baseline` vs `B_location_spatial`.

This is the conservative setting because both models already include `읍면동`; therefore, any improvement from `B_location_spatial` means spatial variables add value beyond administrative dong information.

In [ ]:
RUN_STRICT_VALIDATION = True
STRICT_GROUPS = ["A_location_baseline", "B_location_spatial"]


def run_strict_holdout(validation_name, train_idx, test_idx):
    rows = []
    for group_name in STRICT_GROUPS:
        print("Strict", validation_name, group_name)
        metrics, _ = fit_xgb_grid(df, group_name, FEATURE_GROUPS[group_name], train_idx, test_idx)
        metrics["validation"] = validation_name
        rows.append(metrics)
    return rows

if RUN_STRICT_VALIDATION:
    strict_rows = []

    train_idx_s, test_idx_s = train_test_split(df.index, test_size=0.2, random_state=RND)
    strict_rows.extend(run_strict_holdout("random_split", train_idx_s, test_idx_s))

    years = sorted(df["계약연도"].dropna().unique())
    if len(years) >= 2:
        latest_year = years[-1]
        train_idx_s = df.index[df["계약연도"] < latest_year]
        test_idx_s = df.index[df["계약연도"] == latest_year]
        if len(train_idx_s) >= 100 and len(test_idx_s) >= 50:
            strict_rows.extend(run_strict_holdout(f"time_holdout_test_{latest_year}", train_idx_s, test_idx_s))

    complex_groups = df["단지명"].fillna("missing_complex")
    train_pos, test_pos = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RND).split(df, groups=complex_groups))
    strict_rows.extend(run_strict_holdout("complex_holdout", df.index[train_pos], df.index[test_pos]))

    dong_groups = df["읍면동"].fillna("missing_dong")
    train_pos, test_pos = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RND).split(df, groups=dong_groups))
    strict_rows.extend(run_strict_holdout("dong_holdout", df.index[train_pos], df.index[test_pos]))

    strict_results = pd.DataFrame(strict_rows).sort_values(["validation", "experiment"])
    display(strict_results[[
        "validation", "experiment", "model", "n_train", "n_test", "cv_rmse",
        "test_rmse", "test_mae", "test_mape_pct", "test_r2", "best_params"
    ]])
    strict_results.to_csv(OUT_DIR / "strict_validation_results.csv", index=False, encoding="utf-8-sig")

    strict_wide = strict_results.pivot_table(
        index="validation",
        columns="experiment",
        values=["test_rmse", "test_mae", "test_mape_pct", "test_r2"],
        aggfunc="first",
    )
    left, right = STRICT_GROUPS
    strict_delta = pd.DataFrame(index=strict_wide.index)
    strict_delta["delta_rmse"] = strict_wide[("test_rmse", right)] - strict_wide[("test_rmse", left)]
    strict_delta["delta_mae"] = strict_wide[("test_mae", right)] - strict_wide[("test_mae", left)]
    strict_delta["delta_mape_pct"] = strict_wide[("test_mape_pct", right)] - strict_wide[("test_mape_pct", left)]
    strict_delta["delta_r2"] = strict_wide[("test_r2", right)] - strict_wide[("test_r2", left)]
    strict_delta["spatial_improves_rmse"] = strict_delta["delta_rmse"] < 0
    strict_delta["spatial_improves_mape"] = strict_delta["delta_mape_pct"] < 0
    strict_delta["spatial_improves_r2"] = strict_delta["delta_r2"] > 0

    print(f"Strict delta: {right} - {left}")
    display(strict_delta)
    strict_delta.to_csv(OUT_DIR / "strict_validation_delta.csv", encoding="utf-8-sig")
else:
    print("Strict validation skipped.")

## 14. Reporting Guide

Use the following logic in the report:

1. Do not emphasize Ridge improvement. Ridge is no longer part of the final model story.
2. State that XGBoost was selected as the final model due to best predictive accuracy.
3. Use `B_location_spatial` as the final performance model.
4. Use SHAP to decide how strongly to emphasize spatial-derived variables.
5. If spatial SHAP share is high, write that spatial variables contribute meaningfully.
6. If spatial SHAP share is low, write that XGBoost improves predictive performance, while spatial variables provide only limited additional interpretability after administrative location control.
7. Because monthly-rent conversion was accepted as valid, the expanded-sample result can be used as the main result. If needed, Jeonse-only results can remain as a supplementary robustness check.